# 04 — Encounter-Level Cohort Integration

**Project:** Predicting Opioid Dispensing in Low Back Pain Emergency Visits  
**Phase:** 04 — Encounter-Level Cohort Integration  
**Dataset:** MIMIC-IV-ED v2.2, linked MIMIC-IV patient data  
**Unit of Analysis:** ED encounter (`stay_id`)

## Executive Summary

This notebook constructs the encounter-level analytical dataset for a retrospective machine learning study of documented opioid dispensing among emergency department (ED) encounters with an ICD-defined low back pain (LBP) phenotype.

Building on the completed data quality assessment (P1), ICD-based clinical phenotyping (P2), and chief complaint feature engineering (P3), this phase integrates retrospective cohort membership, adult eligibility, structured triage information, and previously developed chief complaint features.

Two nested ICD-based cohorts are used: C1 (Strict LBP), the primary analytical cohort, and C2 (Expanded LBP), the sensitivity cohort. Final ICD diagnoses determine retrospective cohort membership but are excluded from the triage-time predictive features.

The primary output is a reproducible, adult encounter-level dataset with one record per `stay_id`, documented eligibility criteria, and validated data integration.

Opioid dispensing outcome construction and machine learning preprocessing are deferred to P5 and P6, respectively.

## Table of Contents

1. [Study Context & Data Sources](#1-study-context--data-sources)
2. [ICD Mapping & Cohort Extraction](#2-icd-mapping--cohort-extraction)
3. [Adult Eligibility & Encounter Integrity](#3-adult-eligibility--encounter-integrity)
4. [Triage & Chief Complaint Integration](#4-triage--chief-complaint-integration)
5. [Cohort Characteristics & Data Quality](#5-cohort-characteristics--data-quality)
6. [Final Dataset Contract & P5 Handoff](#6-final-dataset-contract--p5-handoff)

## 1. Study Context & Data Sources

### 1.1 Research Context

This notebook supports the project *Predicting Opioid Dispensing in Low Back Pain Emergency Visits: Explainable Machine Learning and Clinical Phenotype Sensitivity*.

The study investigates whether routinely available ED triage information can predict documented opioid dispensing among encounters retrospectively identified using ICD-defined LBP phenotypes.

Two prespecified phenotype definitions were finalized in P2:

- **C1 — Strict LBP:** Five ICD-9/10 codes representing explicit low back pain, including LBP with sciatica.
- **C2 — Expanded LBP:** Twelve ICD-9/10 codes, including all C1 codes and additional sciatica or broader back pain symptom codes.

C1 is nested within C2. Encounters identified exclusively through C2 will be retained as a distinct analytical subgroup (`C2-only`).

These phenotypes are operational, literature-informed definitions rather than independently clinically validated diagnoses.

The purpose of P4 is to translate these finalized definitions into reproducible encounter-level analytical cohorts while integrating the triage information required for downstream predictive modeling.

### 1.2 Data Sources

**Dataset:** MIMIC-IV-ED v2.2 (Johnson et al., 2023)  
**Query platform:** Google BigQuery  
**Unit of analysis:** ED encounter (`stay_id`)

| Source | Role in the Study |
|---|---|
| `edstays` | Encounter identifiers, ED timing, demographics, and arrival transport |
| `diagnosis` | Retrospective ICD-based cohort identification |
| `triage` | Structured triage observations |
| MIMIC-IV `patients` | Age estimation and adult eligibility |
| `icd_lbp_codebook_v1.csv` | Finalized C1/C2 ICD mapping from P2 |
| `chief_complaint_features.csv` | Normalized complaint text and frozen clinical indicators from P3 |

Previously completed P1 analyses establish the source-table structure and baseline data-quality considerations. P4 will focus on cohort-specific integration checks rather than repeating database-wide exploratory analysis.

### 1.3 Objectives and Scope

This notebook aims to:

1. Apply the finalized P2 ICD codebook to identify C1, C2, and C2-only encounters.
2. Establish adult eligibility using linked MIMIC-IV patient information.
3. Aggregate multiple diagnosis records into unique encounter-level cohort indicators.
4. Integrate structured triage information and frozen P3 chief complaint features.
5. Assess cohort composition, join completeness, feature missingness, and encounter-level integrity.
6. Produce a validated analytical dataset for P5 opioid outcome integration and P6 predictive modeling.

The analytical dataset will preserve `subject_id` to support patient-level train/test partitioning in P6.

Final ICD diagnoses, ED disposition, encounter duration, and medication-related information will not be included as triage-time predictive features.

This phase does not construct opioid outcome labels, perform train/test splitting, or fit machine learning preprocessing transformations.

### 1.4 Analytical Workflow and Outputs

BigQuery SQL scripts are maintained under `sql/04_cohort_integration/`. Query results are exported as local CSV files under `results/04_cohort_integration/`.

The workflow consists of:

1. **Cohort extraction:** Match encounter diagnosis records against the versioned ICD codebook and aggregate C1/C2 membership by `stay_id`.
2. **Eligibility and structured data:** Link ED encounters to patient information for adult eligibility and retain candidate triage predictors.
3. **Local feature integration:** Join the BigQuery encounter extract with the previously exported P3 chief complaint features using Pandas.
4. **Quality assurance:** Validate unique encounter identifiers, cohort nesting, eligibility, join completeness, and missing predictor values.
5. **Final export:** Save the validated C2 analytical dataset, with C1 represented as a nested subset.

No patient-level MIMIC data will be committed to the public or private project Git repository.

The primary output is a validated encounter-level dataset containing cohort membership indicators, patient identifiers, structured triage variables, and chief complaint representations.

### Reference

Johnson, A. E. W., et al. (2023). *MIMIC-IV-ED, version 2.2*. PhysioNet. https://physionet.org/content/mimic-iv-ed/2.2/

In [ ]:

from pathlib import Path

import pandas as pd
import numpy as np

# Project root: works when launched from the repo or notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

CODEBOOK_DIR = PROJECT_ROOT / "codebooks"
RESULTS_DIR = PROJECT_ROOT / "results"
P4_RESULTS_DIR = RESULTS_DIR / "04_cohort_integration"

CODEBOOK_PATH = CODEBOOK_DIR / "icd_lbp_codebook_v1.csv"
P3_FEATURE_PATH = RESULTS_DIR / "03_chief_complaint" / "chief_complaint_features.csv"

P4_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Codebook exists: {CODEBOOK_PATH.exists()}")
print(f"P3 features exist: {P3_FEATURE_PATH.exists()}")


Project root: /Users/elisali/Desktop/MSAI/AI_380H_Healthcare/HRP/lbp-opioid-prediction
Codebook exists: True
P3 features exist: True


## 2. ICD Mapping & Cohort Extraction
### 2.1 Final ICD Codebook Verification

**Objective**

Verify the finalized P2 ICD codebook before applying it to encounter-level diagnosis records.

**Method**

Load the versioned codebook and perform a focused data-contract check:

- Confirm the expected columns and unique (`icd_version`, `icd_code`) combinations.
- Validate the C1/C2 inclusion flags and their nested relationship.
- Verify the expected mapping of five C1 codes and twelve C2 codes.
- Preserve ICD codes as strings to ensure consistent matching with MIMIC-IV-ED diagnosis records.

The codebook was clinically reviewed and finalized in P2. This section verifies its technical suitability for downstream cohort construction without repeating clinical code selection or phenotype validation.

**Expected Output**

A verified ICD lookup containing the composite key and binary C1/C2 inclusion flags, ready for BigQuery integration.

In [2]:

# 2.1 Load and verify the finalized ICD codebook

codebook = pd.read_csv(
    CODEBOOK_PATH,
    dtype={"icd_version": "string", "icd_code": "string"}
)

required_cols = [
    "icd_version", "icd_code", "c1_include", "c2_include"
]

assert set(required_cols).issubset(codebook.columns)
assert not codebook[required_cols].isna().any().any()
assert not codebook.duplicated(
    ["icd_version", "icd_code"]
).any()

# Normalize inclusion flags to binary integers
for col in ["c1_include", "c2_include"]:
    values = codebook[col].astype(str).str.strip().str.lower()
    assert values.isin(["true", "false", "1", "0"]).all()
    codebook[col] = values.map({
        "true": 1, "false": 0, "1": 1, "0": 0
    }).astype("int8")

# Validate version and cohort nesting
assert set(codebook["icd_version"]) <= {"9", "10"}
assert (codebook["c1_include"] <= codebook["c2_include"]).all()

# Confirm the frozen P2 mapping contract
assert len(codebook) == 12
assert codebook["c1_include"].sum() == 5
assert codebook["c2_include"].sum() == 12

# Minimal SQL lookup
icd_lookup = codebook[required_cols].copy()

print("ICD codebook verification: PASSED")
print(f"Unique ICD pairs: {len(icd_lookup)}")
print(f"C1 codes: {icd_lookup['c1_include'].sum()}")
print(f"C2 codes: {icd_lookup['c2_include'].sum()}")

display(
    icd_lookup.sort_values(["icd_version", "icd_code"])
    .reset_index(drop=True)
)


ICD codebook verification: PASSED
Unique ICD pairs: 12
C1 codes: 5
C2 codes: 12


,icd_version,icd_code,c1_include,c2_include
0,10,M5430,0,1
1,10,M5431,0,1
2,10,M5432,0,1
3,10,M5440,1,1
4,10,M5441,1,1
5,10,M5442,1,1
6,10,M545,1,1
7,10,M5489,0,1
8,10,M549,0,1
9,9,7242,1,1


**Findings**

The finalized P2 ICD codebook passed all technical verification checks. It contains 12 unique ICD code/version pairs, including five C1 (Strict LBP) codes and twelve C2 (Expanded LBP) codes.

No duplicate composite keys, missing mapping fields, or cohort nesting violations were identified.

The verified mapping will be used for encounter-level phenotype identification. No additional ICD code selection or clinical validation is performed in P4.

### 2.2 BigQuery ICD Lookup Preparation

**Objective**

Make the verified P2 ICD codebook available in BigQuery as a reusable lookup table for encounter-level phenotype identification.

**Method**

Export the four required mapping columns from the verified codebook and load them into a dedicated BigQuery reference table.

The lookup uses (`icd_version`, `icd_code`) as its composite matching key and retains the two binary cohort indicators (`c1_include`, `c2_include`).

The BigQuery table is a derived representation of the versioned P2 CSV, not an independently maintained codebook.

**Expected Output**

A BigQuery ICD lookup table containing 12 unique code/version pairs, with five C1-positive and twelve C2-positive codes.

### 2.2.1 Export Verified ICD Mapping

**Objective**

Prepare a minimal BigQuery-compatible lookup file from the finalized and technically verified P2 ICD codebook.

**Method**

The verified `icd_lookup` DataFrame is exported to CSV using four fields:

- `icd_version`
- `icd_code`
- `c1_include`
- `c2_include`

The exported file is reloaded to verify that its contents remain consistent with the in-memory lookup. ICD codes and versions are preserved as strings.

**Output**

`results/04_cohort_integration/icd_lbp_bigquery_lookup.csv`

The resulting lookup contains 12 ICD code/version pairs, including five C1 codes and twelve C2 codes. It serves as the input for the BigQuery reference table.

In [3]:

# 2.2.1 Export verified ICD mapping for BigQuery

lookup_path = P4_RESULTS_DIR / "icd_lbp_bigquery_lookup.csv"

icd_lookup.to_csv(lookup_path, index=False)

# Verify exported file
lookup_check = pd.read_csv(
    lookup_path,
    dtype={"icd_version": "string", "icd_code": "string"}
)

pd.testing.assert_frame_equal(
    lookup_check,
    icd_lookup.reset_index(drop=True),
    check_dtype=False
)

print(f"Exported: {lookup_path}")
print(f"Rows: {len(lookup_check)}")
print("Lookup CSV verification: PASSED")


Exported: /Users/elisali/Desktop/MSAI/AI_380H_Healthcare/HRP/lbp-opioid-prediction/results/04_cohort_integration/icd_lbp_bigquery_lookup.csv
Rows: 12
Lookup CSV verification: PASSED


### 2.2.2 Upload ICD Lookup to BigQuery

The verified lookup CSV was uploaded to BigQuery as
`icd_lbp_codebook_v1`, using explicitly defined schema types.
The uploaded table was subsequently validated in Section 2.2.3.

### 2.2.3 BigQuery Lookup Verification

**Objective**

Confirm that the uploaded BigQuery ICD lookup preserves the finalized P2 mapping without introducing duplicate keys or invalid cohort membership flags.

**Method**

The verified CSV was uploaded to BigQuery as `icd_lbp_codebook_v1`, with explicit data types for ICD codes, versions, and inclusion flags.

The verification query (`sql/04_cohort_integration/01_verify_icd_lookup.sql`) assessed:

- Total mapping records and unique ICD code/version pairs.
- C1 and C2 inclusion counts.
- Cohort nesting violations.
- Missing mapping fields.

**Results**

The BigQuery verification query executed successfully and confirmed:

| Validation Measure | Result |
|---|---:|
| Total ICD mappings | 12 |
| Unique ICD code/version pairs | 12 |
| C1 codes | 5 |
| C2 codes | 12 |
| Cohort nesting violations | 0 |
| Missing mapping records | 0 |

**Interpretation**

The uploaded lookup table is consistent with the finalized P2 codebook and passed all technical verification checks. It provides a reusable ICD mapping for encounter-level cohort construction without introducing a new phenotype definition.

### 2.3 Encounter-Level ICD Cohort Extraction

**Objective**

Identify ED encounters meeting the finalized C1 (Strict LBP) and C2 (Expanded LBP) phenotype definitions and construct one set of cohort membership indicators per encounter.

**Method**

The MIMIC-IV-ED `diagnosis` table is matched against the verified BigQuery ICD lookup using the composite key (`icd_version`, `icd_code`).

Because an ED encounter may have multiple diagnosis records, matched diagnoses are aggregated by `stay_id` before integration with `edstays`.

Three binary indicators are constructed:

- `c1_flag`: At least one diagnosis matches the strict LBP phenotype.
- `c2_flag`: At least one diagnosis matches the expanded LBP phenotype.
- `c2_only_flag`: The encounter matches C2 but not C1.

All diagnosis positions are eligible for phenotype identification. The presence of additional diagnoses does not independently exclude an encounter.

Only C2-positive encounters are retained in the extraction, with C1 represented as a nested subset.

**Expected Output**

An encounter-level ICD cohort extract containing:

| Variable | Description |
|---|---|
| `stay_id` | Unique ED encounter identifier |
| `subject_id` | Patient identifier |
| `c1_flag` | Strict LBP membership |
| `c2_flag` | Expanded LBP membership |
| `c2_only_flag` | Expanded-only membership |

The output includes encounters of all ages. Adult eligibility will be applied separately in Section 3.

Cohort counts and encounter-level integrity will be verified using actual extraction results.

### 2.3.1 ICD Cohort Extraction in BigQuery

**Objective**

Identify ED encounters matching the finalized ICD-based LBP phenotypes while preserving one record per encounter.

**Method**

The BigQuery script `sql/04_cohort_integration/02_extract_icd_cohort.sql` performs the following operations:

1. Matches diagnosis records to the verified P2 ICD lookup using (`icd_version`, `icd_code`).
2. Aggregates matched diagnosis records by `stay_id` to generate `c1_flag` and `c2_flag`.
3. Joins the aggregated flags with `edstays` to retrieve `subject_id`.
4. Derives `c2_only_flag` and retains C2-positive encounters.

**Output**

The extraction produces five fields: `stay_id`, `subject_id`, `c1_flag`, `c2_flag`, and `c2_only_flag`.

Adult eligibility and structured triage features are not applied at this stage.

### 2.3.2 Query Execution and Data Export

The cohort extraction query was executed in Google BigQuery, and its output was exported to:

`results/04_cohort_integration/icd_cohort_encounters_raw.csv`

The export contains **14,299 ICD-defined C2-positive ED encounters**, including both C1 and C2-only encounters.

The CSV is retained locally and excluded from Git version control. It serves as the input for the encounter-level validation in Section 2.3.3.

No adult eligibility filter has been applied to this extract.

### 2.3.3 Encounter-Level Cohort Validation

**Objective**

Validate the encounter-level ICD cohort extract and summarize membership in the two nested phenotype definitions.

**Method**

The BigQuery extract (`02_extract_icd_cohort.sql`) is evaluated for:

- Unique and non-null encounter and patient identifiers.
- Valid binary cohort indicators and consistent C1/C2 nesting.
- Correct classification of C2-only encounters.
- Encounter and unique patient counts for C1, C2, and C2-only.

These checks verify the integrity of the extracted cohort records, not the clinical validity of the ICD phenotype definitions.

**Expected Output**

A validated encounter-level dataset with documented cohort counts and no duplicate `stay_id` records. Adult eligibility has not yet been applied.

In [4]:

# 2.3.3 Validate encounter-level ICD cohort extraction

cohort_path = P4_RESULTS_DIR / "icd_cohort_encounters_raw.csv"
cohort = pd.read_csv(cohort_path)

required = [
    "stay_id", "subject_id",
    "c1_flag", "c2_flag", "c2_only_flag"
]

# Encounter integrity
assert set(required).issubset(cohort.columns)
assert cohort[required].notna().all().all()
assert cohort["stay_id"].is_unique

# Cohort flag integrity
for col in ["c1_flag", "c2_flag", "c2_only_flag"]:
    assert cohort[col].isin([0, 1]).all()

assert (cohort["c2_flag"] == 1).all()
assert (cohort["c1_flag"] <= cohort["c2_flag"]).all()
assert (
    cohort["c2_only_flag"] == 1 - cohort["c1_flag"]
).all()

# Summarize encounter and patient counts
groups = {
    "C1 — Strict LBP": cohort["c1_flag"] == 1,
    "C2-only": cohort["c2_only_flag"] == 1,
    "C2 — Expanded LBP": cohort["c2_flag"] == 1,
}

summary = pd.DataFrame([
    {
        "cohort": name,
        "encounters": mask.sum(),
        "unique_patients": cohort.loc[mask, "subject_id"].nunique()
    }
    for name, mask in groups.items()
])

print("Encounter-level ICD cohort QA: PASSED")
print(f"Total encounters: {len(cohort):,}")
print(f"Unique patients: {cohort['subject_id'].nunique():,}")

display(summary)


Encounter-level ICD cohort QA: PASSED
Total encounters: 14,299
Unique patients: 11,409


,cohort,encounters,unique_patients
0,C1 — Strict LBP,9576,7947
1,C2-only,4723,4275
2,C2 — Expanded LBP,14299,11409


**Findings**

The encounter-level ICD cohort extraction passed all data integrity checks.

The expanded C2 phenotype identified **14,299 ED encounters among 11,409 unique patients**. Of these, 9,576 encounters met the strict C1 definition, while 4,723 were identified exclusively through the expanded C2 definition.

All encounter identifiers were unique and non-null, all cohort indicators were internally consistent, and no C1/C2 nesting violations were observed.

The expanded definition captured approximately 49.3% more encounters than C1 alone. This demonstrates a meaningful difference in cohort coverage between the two operational ICD definitions, although its implications for patient characteristics and opioid dispensing remain to be evaluated.

These counts represent the ICD-defined population before adult eligibility filtering and should not yet be reported as the final analytical cohort.

## 3. Adult Eligibility & Encounter Integrity

### 3.1 Patient Linkage & Age Estimation

**Objective**

Assess the completeness of patient-level linkage and the feasibility of estimating age at ED arrival before applying adult eligibility criteria.

**Method**

The ICD-defined C2 encounters extracted in Section 2 are linked to MIMIC-IV `patients` using `subject_id`.

Age at ED arrival is estimated using the MIMIC-IV deidentification anchors:

\[
\text{Estimated Age} = \text{anchor\_age} + (\text{ED arrival year} - \text{anchor\_year})
\]

The following checks are performed:

- Patient linkage completeness.
- Missing age anchors and ED arrival timestamps.
- Estimated age distribution and implausible values.
- Number of encounters below and above the adult eligibility threshold (≥18 years).
- Potential ambiguity around the age-18 boundary.

MIMIC-IV applies special deidentification to older patients, so estimated ages may not represent exact chronological ages.

**Expected Output**

A cohort-level linkage and age eligibility summary, including the number of encounters with estimable ages and the number potentially eligible for the adult analytical population.

The eligibility filter will be applied in Section 3.2, after reviewing these results.

### 3.1.1 BigQuery SQL — Age & Linkage Audit
**SQL Script:** `sql/04_cohort_integration/03_check_patient_age.sql`  
**Output:** `results/04_cohort_integration/patient_age_linkage_audit.csv`

**Method**

The ICD-defined C2 encounters were reconstructed using the verified P2 codebook and linked to the MIMIC-IV `patients` table through `subject_id` using a left join.

Estimated age at ED arrival was calculated from `anchor_age`, `anchor_year`, and the ED arrival year. The query assessed encounter uniqueness, patient linkage completeness, missing age information, implausible ages, and the distribution around the adult eligibility threshold.

**Execution**

The query was executed in Google BigQuery and returned a single aggregated QA record. Detailed results and interpretation are presented in Section 3.1.2.

### 3.1.2 Patient Linkage and Age Audit Results

**Objective**

Verify the exported BigQuery age-linkage audit results and assess completeness of adult eligibility information.

**Method**

The aggregated audit output (`results/04_cohort_integration/patient_age_linkage_audit.csv`) is loaded using Pandas.

The analysis checks:

- Encounter uniqueness and patient linkage completeness.
- Missing ED arrival timestamps and age estimates.
- Adult eligibility and age-18 boundary counts.
- Estimated age range and high-age deidentification indicators.

Basic consistency assertions are applied to the exported results. This step verifies the existing BigQuery audit without repeating its underlying SQL calculations.

**Expected Output**

A reproducible summary of patient linkage and age eligibility, supporting the adult cohort construction in Section 3.2.

In [5]:

# 3.1.2 Verify and display the BigQuery age-linkage audit

age_audit_path = P4_RESULTS_DIR / "patient_age_linkage_audit.csv"
age_audit = pd.read_csv(age_audit_path)

# Verify the expected audit structure
required_metrics = [
    "total_rows", "unique_stays", "unique_patients",
    "unmatched_patient_rows", "missing_intime",
    "missing_age_anchors", "missing_estimated_age",
    "negative_age", "under_18", "age_18",
    "age_18_or_older", "anchor_age_91",
    "min_estimated_age", "max_estimated_age",
    "mean_estimated_age"
]

assert len(age_audit) == 1
assert set(required_metrics).issubset(age_audit.columns)
assert age_audit[required_metrics].notna().all().all()

audit = age_audit.iloc[0]

# Core consistency checks
assert audit["total_rows"] == audit["unique_stays"]
assert audit["total_rows"] == len(cohort)
assert audit["missing_estimated_age"] == (
    audit["unmatched_patient_rows"]
    + audit["missing_age_anchors"]
)
assert (
    audit["under_18"]
    + audit["age_18_or_older"]
    + audit["missing_estimated_age"]
    == audit["total_rows"]
)
assert audit["age_18"] <= audit["age_18_or_older"]
assert audit["negative_age"] == 0

# Display audit results
age_audit_summary = pd.DataFrame({
    "Measure": required_metrics,
    "Value": [audit[col] for col in required_metrics]
})

print("Patient age-linkage audit QA: PASSED")
print(
    f"Patient linkage completeness: "
    f"{(1 - audit['unmatched_patient_rows'] / audit['total_rows']) * 100:.2f}%"
)

display(age_audit_summary)


Patient age-linkage audit QA: PASSED
Patient linkage completeness: 99.96%


,Measure,Value
0,total_rows,14299.00
1,unique_stays,14299.00
2,unique_patients,11409.00
3,unmatched_patient_rows,6.00
4,missing_intime,0.00
5,missing_age_anchors,0.00
6,missing_estimated_age,6.00
7,negative_age,0.00
8,under_18,0.00
9,age_18,31.00


**Findings**

The age-linkage audit passed all consistency checks. Of 14,299 ICD-defined C2 encounters, 14,293 (99.96%) were successfully linked to patient age information and had estimated ages ≥18 years.

Six encounters lacked estimable ages because their patient records could not be matched. No encounters with available age information were below 18 years.

The estimated age range was 18–99 years, with 31 encounters at the age-18 boundary. An additional 139 encounters had `anchor_age = 91`, reflecting MIMIC-IV high-age deidentification.

**Interpretation**

Age information was available for nearly all ICD-defined encounters. The six unresolved encounters will be excluded from the adult analytical population because eligibility cannot be confirmed.

Estimated age is derived from deidentification anchors rather than exact birth dates. The results support the prespecified adult eligibility criteria applied in Section 3.2.

### 3.1.3 Unmatched Patient Linkage Review

**SQL Script:** `sql/04_cohort_integration/04_audit_unmatched_patients.sql`

**Objective**

Characterize encounters without a matching patient record and determine their distribution across the ICD-defined cohorts before applying adult eligibility criteria.

**Method**

A focused BigQuery query identified C2-positive encounters whose `subject_id` did not match a record in the linked MIMIC-IV `patients` table.

The query summarized unmatched encounter counts, unique patients, and C1 versus C2-only membership.

**Results**

Six unmatched ED encounters were identified, involving five unique patients:

| Measure | Count |
|---|---:|
| Unmatched encounters | 6 |
| Unique encounters | 6 |
| Unique patients | 5 |
| C1 encounters | 5 |
| C2-only encounters | 1 |

**Interpretation**

The unmatched encounters represent a small proportion of the ICD-defined population (6/14,299; approximately 0.04%).

Because adult eligibility cannot be established for these encounters, these six encounters were excluded from the adult analytical cohort.

No additional clinical or ICD-based exclusion criteria are introduced.

### 3.2 Adult Eligibility & Cohort Construction

**Objective**

Apply the prespecified adult eligibility criterion to the ICD-defined LBP encounters and establish the final adult C1/C2 analytical population.

**Method**

Based on the linkage and age audit in Section 3.1, encounters are eligible if they:

- Meet the expanded C2 ICD phenotype definition.
- Have a valid ED encounter identifier (`stay_id`).
- Have an estimated age of at least 18 years.

Age is estimated using MIMIC-IV patient age anchors and the ED arrival year.

Six encounters with unavailable age information are excluded because adult eligibility cannot be established. No additional diagnosis-based, disposition-based, or medication-based exclusion criteria are applied.

The adult cohort retains C1 as a nested subset of C2, with C2-only membership represented separately.

**Expected Output**

A validated adult encounter-level cohort containing patient and encounter identifiers, estimated age, and C1/C2 membership indicators.

The resulting dataset provides the eligibility foundation for structured triage and chief complaint integration in Section 4.

### 3.2.1 Adult Cohort Construction in BigQuery

**SQL Script:** `sql/04_cohort_integration/05_extract_adult_cohort.sql`

**Objective**

Construct the adult ICD-defined low back pain cohort by applying the prespecified age eligibility criterion to the verified encounter-level phenotype definitions.

**Method**

The BigQuery query:

1. Matches diagnosis records to the finalized P2 ICD mapping.
2. Aggregates diagnosis-level matches into C1 and C2 encounter flags.
3. Links ED encounters to MIMIC-IV patient age anchors using `subject_id`.
4. Estimates age at ED arrival using `anchor_age`, `anchor_year`, and ED arrival year.
5. Retains C2-positive encounters with estimated age ≥18 years.

C1 and C2-only indicators are preserved for subsequent phenotype sensitivity analysis.

Encounters without estimable age are excluded from the adult analytical population. No additional clinical or medication-based exclusions are introduced.

**Output**

An encounter-level adult cohort containing patient and encounter identifiers, estimated age, and C1/C2 membership indicators.

### 3.2.2 Adult Cohort Extraction and Export

**SQL Script:** `sql/04_cohort_integration/05_extract_adult_cohort.sql`  
**Output:** `results/04_cohort_integration/adult_icd_cohort.csv`

**Execution**

The adult eligibility criteria were applied in Google BigQuery using the finalized ICD cohort mapping and linked MIMIC-IV patient age anchors.

The extraction retained C2-positive encounters with estimated age ≥18 years and preserved C1 and C2-only membership indicators.

Six encounters with unavailable age information were excluded from the analytical population, as documented in Section 3.1.

The exported dataset contained **14,293 adult ED encounters** and was subsequently validated in Section 3.2.3 for encounter uniqueness, cohort consistency, adult eligibility, and patient counts.

### 3.2.3 Final Adult Cohort QA & Findings

**Objective**

Verify that the exported adult ICD-defined cohort satisfies the eligibility and encounter-level integrity requirements before integrating triage features.

**Method**

The exported `adult_icd_cohort.csv` is checked for:

- Unique and non-null encounter and patient identifiers.
- Estimated age ≥18 years.
- Valid binary C1/C2 cohort indicators.
- Correct C1 nesting and C2-only classification.
- Encounter and unique patient counts across the three cohort groups.

The validation uses the final exported CSV rather than repeating the BigQuery extraction.

**Expected Output**

A validated adult C2 cohort containing exactly one record per `stay_id`, with C1 represented as a nested subset and C2-only encounters identified separately.

In [6]:

# 3.2.3 Validate final adult ICD cohort

adult_path = P4_RESULTS_DIR / "adult_icd_cohort.csv"
adult = pd.read_csv(adult_path)

required = [
    "stay_id", "subject_id", "estimated_age",
    "c1_flag", "c2_flag", "c2_only_flag"
]

assert set(required).issubset(adult.columns)
assert adult[required].notna().all().all()
assert adult["stay_id"].is_unique
assert (adult["estimated_age"] >= 18).all()

for col in ["c1_flag", "c2_flag", "c2_only_flag"]:
    assert adult[col].isin([0, 1]).all()

assert (adult["c2_flag"] == 1).all()
assert (adult["c1_flag"] <= adult["c2_flag"]).all()
assert (
    adult["c2_only_flag"] == adult["c2_flag"] - adult["c1_flag"]
).all()

groups = {
    "C1 — Strict LBP": adult["c1_flag"] == 1,
    "C2-only": adult["c2_only_flag"] == 1,
    "C2 — Expanded LBP": adult["c2_flag"] == 1,
}

adult_summary = pd.DataFrame([
    {
        "cohort": name,
        "encounters": int(mask.sum()),
        "unique_patients": adult.loc[mask, "subject_id"].nunique()
    }
    for name, mask in groups.items()
])

assert (
    adult_summary.loc[0, "encounters"]
    + adult_summary.loc[1, "encounters"]
    == adult_summary.loc[2, "encounters"]
)

print("Adult cohort integrity QA: PASSED")
print(f"Total adult encounters: {len(adult):,}")
print(f"Unique adult patients: {adult['subject_id'].nunique():,}")
print(f"Estimated age range: {adult['estimated_age'].min()}–{adult['estimated_age'].max()}")

display(adult_summary)


Adult cohort integrity QA: PASSED
Total adult encounters: 14,293
Unique adult patients: 11,404
Estimated age range: 18–99


,cohort,encounters,unique_patients
0,C1 — Strict LBP,9571,7942
1,C2-only,4722,4274
2,C2 — Expanded LBP,14293,11404


**Findings**

The adult cohort extraction identified **14,293 eligible ED encounters among 11,404 unique patients**.

The strict C1 phenotype included 9,571 encounters, while 4,722 encounters were identified exclusively through the expanded C2 phenotype.

Six encounters from the original ICD-defined population were excluded because age eligibility could not be established through the linked patient records. No encounters with estimable ages were below 18 years.

The final adult cohort contains unique encounter identifiers, valid nested phenotype indicators, and estimated ages ≥18 years. The cohort retains all ED dispositions and does not exclude encounters based on medication-dispensing records.

These results establish the adult ICD-defined analytical population for subsequent triage feature integration. The phenotype definitions remain retrospective operational classifications and have not undergone independent clinical validation.

## 4. Triage & Chief Complaint Integration

### 4.1 Structured Triage Extraction

**Objective**

Extract routinely available structured triage information for the adult ICD-defined low back pain cohort established in Section 3.

**Method**

The finalized ICD mapping and adult eligibility criteria are applied in BigQuery to identify eligible C2 encounters. These encounters are linked to `edstays` and `triage` using `stay_id`.

Candidate predictors are organized into two groups:

- **Core clinical features:** Estimated age, recorded gender, pain score, and triage acuity.
- **Extended triage features:** Arrival transport, heart rate, systolic/diastolic blood pressure, respiratory rate, temperature, and oxygen saturation.

All candidate features are retained at this stage, regardless of their eventual predictive contribution. Feature-set selection and learned preprocessing will occur in P6.

The extraction uses a left join to `triage` to preserve eligible encounters even when triage information is unavailable. A join indicator is retained to distinguish unmatched triage records from missing individual measurements.

**Expected Output**

A structured triage extract covering all 14,293 eligible adult C2 encounters, subject to integrity validation.

The extraction will preserve one row per `stay_id`. Chief complaint text and rule-based features will be integrated separately using the finalized P3 CSV.

No opioid outcome information or post-triage predictors are included.

### 4.1.1 Structured Triage Extraction in BigQuery

**SQL Script:** `sql/04_cohort_integration/06_extract_structured_triage.sql`

The query reconstructs the eligible adult C2 cohort using the verified ICD mapping and patient age anchors, then joins structured triage observations by `stay_id`.

Core and extended candidate predictors are extracted without applying outcome-based feature selection or learned preprocessing.

A `triage_matched` indicator is retained for integration quality assurance.

### 4.1.2 Query Execution and Data Export

**Output:** `results/04_cohort_integration/structured_triage_raw.csv`

The BigQuery output is exported locally for integration with the validated adult cohort and finalized P3 chief complaint features.

Encounter uniqueness, patient identifier consistency, join completeness, and structured feature missingness will be assessed in Section 4.2.

**Findings**

The structured triage extraction produced 14,293 records, matching the finalized adult C2 cohort.

All `stay_id` values were unique and non-null, and every eligible encounter had a matching triage record. The extracted encounter identifiers and patient identifiers were consistent with the adult cohort established in Section 3.

Structured triage measurements showed relatively low raw missingness, ranging from 0.47% for acuity to 1.98% for temperature. However, raw pain completeness does not imply valid numeric pain scores; some nonempty pain values contain out-of-range or nonnumeric expressions.

All raw candidate predictors are retained for downstream modeling. Feature cleaning, learned imputation, and model-specific preprocessing are deferred to P6.

### 4.2 Triage Integration & Integrity QA

**Objective**

Integrate the validated adult ICD-defined cohort with structured ED triage information while preserving encounter-level integrity.

**Method**

The adult cohort from Section 3 is merged with the structured triage extract using `stay_id`.

The integration verifies:

- One-to-one encounter matching without row multiplication.
- Preservation of all eligible adult encounters.
- Consistency of `subject_id` across source tables.
- Complete triage record linkage.
- Missingness of structured candidate predictors.

Raw feature values are preserved. No outcome-based feature selection, imputation, or model fitting is performed.

**Expected Output**

An encounter-level DataFrame containing adult eligibility information, C1/C2 membership indicators, and structured triage candidate predictors, ready for integration with the finalized P3 chief complaint features.

In [7]:

# 4.2 Integrate adult cohort with structured triage

triage_path = P4_RESULTS_DIR / "structured_triage_raw.csv"
triage = pd.read_csv(triage_path)

assert triage["stay_id"].is_unique
assert triage["stay_id"].notna().all()

# Preserve adult cohort as the analytical population
integrated = adult.merge(
    triage,
    on="stay_id",
    how="left",
    validate="one_to_one",
    indicator=True,
    suffixes=("", "_triage")
)

# Encounter and patient integrity
assert len(integrated) == len(adult)
assert integrated["stay_id"].is_unique
assert (integrated["_merge"] == "both").all()
assert (
    integrated["subject_id"]
    == integrated["subject_id_triage"]
).all()
assert (integrated["triage_matched"] == 1).all()

integrated = integrated.drop(
    columns=["subject_id_triage", "_merge"]
)

# Structured feature missingness
structured_features = [
    "gender", "arrival_transport", "acuity", "pain",
    "heartrate", "sbp", "dbp", "resprate",
    "temperature", "o2sat"
]

missing_summary = pd.DataFrame({
    "missing_n": integrated[structured_features].isna().sum(),
    "missing_pct": (
        integrated[structured_features].isna().mean() * 100
    ).round(2)
}).sort_values("missing_pct", ascending=False)

print("Structured triage integration QA: PASSED")
print(f"Integrated encounters: {len(integrated):,}")
print(f"Unique patients: {integrated['subject_id'].nunique():,}")

display(missing_summary)


Structured triage integration QA: PASSED
Integrated encounters: 14,293
Unique patients: 11,404


,missing_n,missing_pct
temperature,283,1.98
o2sat,240,1.68
resprate,217,1.52
dbp,203,1.42
sbp,180,1.26
heartrate,142,0.99
pain,133,0.93
acuity,67,0.47
gender,0,0.00
arrival_transport,0,0.00


### 4.3 Chief Complaint Feature Integration

**Objective**

Integrate the finalized P3 chief complaint text and rule-based clinical indicators into the adult ICD-defined LBP analytical cohort.

**Method**

The previously validated P3 feature export (`chief_complaint_features.csv`) is loaded and joined to the structured encounter-level dataset using `stay_id`.

The integration preserves:

- Raw chief complaint text (`chiefcomplaint`).
- Normalized complaint text (`cc_text_normalized`).
- Missing-text indicator (`cc_text_missing`).
- Four frozen, non-mutually-exclusive clinical text indicators: `cc_explicit_lbp`, `cc_sciatica`, `cc_broad_back_pain`, and `cc_radiating_leg_pain`.

A left join preserves all eligible adult encounters, regardless of whether complaint text is missing or any clinical rule matches.

Join completeness, encounter uniqueness, patient identifiers, and text feature integrity are verified. Missing P3 encounter matches are distinguished from empty complaint text.

The clinical text rules are not modified or re-evaluated against final diagnoses or opioid outcomes.

**Expected Output**

An integrated adult C2 encounter-level dataset containing structured triage candidates and the finalized P3 chief complaint representations, with C1 retained as a nested analytical subset.

In [8]:

# 4.3 Integrate frozen P3 chief complaint features

cc_columns = [
    "stay_id", "chiefcomplaint", "cc_text_normalized",
    "cc_text_missing", "cc_explicit_lbp",
    "cc_sciatica", "cc_broad_back_pain",
    "cc_radiating_leg_pain"
]

cc = pd.read_csv(P3_FEATURE_PATH, usecols=cc_columns)

# Verify P3 source integrity
assert cc["stay_id"].notna().all()
assert cc["stay_id"].is_unique
assert set(cc_columns) == set(cc.columns)

# Preserve adult C2 cohort; track join separately from text missingness
integrated = integrated.merge(
    cc,
    on="stay_id",
    how="left",
    validate="one_to_one",
    indicator="cc_join_status"
)

assert len(integrated) == len(adult)
assert integrated["stay_id"].is_unique

unmatched_p3 = int(
    (integrated["cc_join_status"] != "both").sum()
)

# Validate binary flags only for successfully matched P3 records
cc_flags = [
    "cc_text_missing",
    "cc_explicit_lbp",
    "cc_sciatica",
    "cc_broad_back_pain",
    "cc_radiating_leg_pain"
]

matched = integrated["cc_join_status"] == "both"

for col in cc_flags:
    assert integrated.loc[matched, col].isin([0, 1]).all()

# Preserve the frozen P3 explicit/broad relationship
assert (
    integrated.loc[matched, "cc_explicit_lbp"]
    <= integrated.loc[matched, "cc_broad_back_pain"]
).all()

# Distinguish missing P3 records from empty complaint text
print("Chief complaint integration QA: PASSED")
print(f"Integrated encounters: {len(integrated):,}")
print(f"Unmatched P3 encounters: {unmatched_p3:,}")
print(
    "Matched encounters with empty normalized text:",
    int(integrated.loc[matched, "cc_text_missing"].sum())
)

display(
    integrated.loc[matched, cc_flags]
    .sum()
    .rename("positive_encounters")
    .to_frame()
)

# Do not drop cc_join_status until final QA in Section 4.4


Chief complaint integration QA: PASSED
Integrated encounters: 14,293
Unmatched P3 encounters: 0
Matched encounters with empty normalized text: 3


,positive_encounters
cc_text_missing,3
cc_explicit_lbp,4070
cc_sciatica,17
cc_broad_back_pain,10097
cc_radiating_leg_pain,3


**Findings**

Chief complaint feature integration passed all encounter-level integrity checks. All 14,293 adult C2 encounters were successfully matched to the finalized P3 feature records, with no unmatched encounters or row multiplication.

Only three encounters (0.02%) had empty normalized chief complaint text.

Within the adult C2 population, the frozen clinical text rules identified:

| Feature | Positive Encounters | Percentage |
|---|---:|---:|
| `cc_explicit_lbp` | 4,070 | 28.48% |
| `cc_sciatica` | 17 | 0.12% |
| `cc_broad_back_pain` | 10,097 | 70.64% |
| `cc_radiating_leg_pain` | 3 | 0.02% |

The clinical indicators are non-mutually-exclusive and represent triage text patterns rather than confirmed diagnoses.

The difference between ICD-based cohort membership and chief complaint feature coverage illustrates the distinction between retrospective diagnosis-based phenotyping and information documented at triage. It does not independently establish phenotype misclassification or clinical validity.

All P3 feature definitions remain unchanged for downstream modeling.

### 4.4 Final Integrated Dataset QA & Export

**Objective**

Validate and export the integrated adult ICD-defined LBP dataset for downstream opioid outcome construction and predictive modeling.

**Method**

The final dataset is checked for:

- Unique, non-null `stay_id` and `subject_id`.
- Adult eligibility and consistent C1/C2 cohort flags.
- Complete structured triage and P3 feature linkage.
- Valid binary clinical text indicators.
- Preservation of all 14,293 eligible adult encounters.

Raw clinical measurements and chief complaint text are retained without learned preprocessing or outcome-based feature selection.

Integration-specific QA fields are removed before export. Cohort membership indicators are retained for descriptive analysis and cohort selection but will not enter the predictive feature matrix. The exported CSV is independently reloaded with explicit data types for identifiers, cohort indicators, clinical text flags, and raw text fields. Schema consistency and key cohort integrity constraints are verified before the dataset is handed off to P5.

**Expected Output**

A single validated encounter-level CSV containing the adult C2 cohort, with C1 represented as a nested subset, ready for P5 opioid outcome integration.

In [9]:

# 4.4 Final dataset integrity and export

assert integrated["stay_id"].is_unique
assert integrated[["stay_id", "subject_id"]].notna().all().all()
assert (integrated["estimated_age"] >= 18).all()

assert (integrated["c2_flag"] == 1).all()
assert (integrated["c1_flag"] <= integrated["c2_flag"]).all()
assert (
    integrated["c2_only_flag"]
    == integrated["c2_flag"] - integrated["c1_flag"]
).all()

assert (integrated["triage_matched"] == 1).all()
assert (integrated["cc_join_status"] == "both").all()

assert len(integrated) == len(adult)

# Remove integration-only QA columns
final_data = integrated.drop(
    columns=["triage_matched", "cc_join_status"]
).copy()

# Verify binary clinical indicators
for col in [
    "cc_text_missing", "cc_explicit_lbp",
    "cc_sciatica", "cc_broad_back_pain",
    "cc_radiating_leg_pain"
]:
    assert final_data[col].isin([0, 1]).all()

# Save the validated P4 analytical foundation
final_path = P4_RESULTS_DIR / "adult_lbp_encounter_features.csv"
final_data.to_csv(final_path, index=False)


# Verify exported dataset schema and integrity

id_cols = ["stay_id", "subject_id"]

flag_cols = [
    "c1_flag", "c2_flag", "c2_only_flag",
    "cc_text_missing", "cc_explicit_lbp",
    "cc_sciatica", "cc_broad_back_pain",
    "cc_radiating_leg_pain"
]

text_cols = [
    "gender", "arrival_transport", "pain",
    "chiefcomplaint", "cc_text_normalized"
]

dtype_map = {
    **{col: "Int64" for col in id_cols},
    **{col: "Int8" for col in flag_cols},
    **{col: "string" for col in text_cols}
}

reloaded = pd.read_csv(
    final_path,
    dtype=dtype_map,
    keep_default_na=False,
    na_values=[""]
)

# Schema and encounter integrity
assert list(reloaded.columns) == list(final_data.columns)
assert len(reloaded) == len(final_data)
assert reloaded["stay_id"].is_unique
assert reloaded[id_cols].notna().all().all()
assert set(reloaded["stay_id"]) == set(final_data["stay_id"])

# Critical data types
assert pd.api.types.is_string_dtype(reloaded["pain"])
assert all(pd.api.types.is_integer_dtype(reloaded[c]) for c in id_cols + flag_cols)

# Cohort and feature integrity
assert (reloaded["estimated_age"] >= 18).all()
assert (reloaded["c2_flag"] == 1).all()
assert (
    reloaded["c2_only_flag"]
    == reloaded["c2_flag"] - reloaded["c1_flag"]
).all()

for col in flag_cols:
    assert reloaded[col].isin([0, 1]).all()

print("P4 final integration QA: PASSED")
print(f"Encounters: {len(reloaded):,}")
print(f"Unique patients: {reloaded['subject_id'].nunique():,}")
print(f"Columns: {len(reloaded.columns)}")
print(f"Exported: {final_path}")

display(reloaded.dtypes.to_frame("dtype"))


P4 final integration QA: PASSED
Encounters: 14,293
Unique patients: 11,404
Columns: 23
Exported: /Users/elisali/Desktop/MSAI/AI_380H_Healthcare/HRP/lbp-opioid-prediction/results/04_cohort_integration/adult_lbp_encounter_features.csv


,dtype
stay_id,Int64
subject_id,Int64
estimated_age,int64
c1_flag,Int8
c2_flag,Int8
c2_only_flag,Int8
gender,string
arrival_transport,string
acuity,float64
pain,string


## 5. Cohort Characteristics & Data Quality
### 5.1 Baseline Characteristics of ICD-Defined Cohorts

**Objective**

Compare baseline demographic and triage characteristics between the strict LBP cohort (C1) and encounters identified exclusively through the expanded ICD phenotype (C2-only).

**Method**

The validated P4 analytical dataset is divided into two mutually exclusive encounter groups:

- **C1 — Strict LBP:** Encounters meeting the primary ICD phenotype.
- **C2-only:** Encounters included exclusively through the expanded phenotype.

Descriptive comparisons include:

- Estimated age (mean and standard deviation).
- Recorded gender (female proportion).
- Arrival transport (ambulance and walk-in proportions).
- Triage acuity (distribution across recorded levels).
- Chief complaint indicators (`cc_explicit_lbp` and `cc_broad_back_pain`).

Categorical proportions are calculated using nonmissing values, with missing or unknown values reported separately where applicable.

The analysis is descriptive. No hypothesis tests, causal interpretations, or outcome-dependent comparisons are performed.

Although C1 and C2-only are mutually exclusive at the encounter level, individual patients may contribute encounters to both groups.

**Expected Output**

A compact baseline characteristics table comparing C1 and C2-only, providing context for downstream phenotype sensitivity analysis.

In [10]:

# 5.1 Baseline characteristics: C1 vs. C2-only

assert final_data["stay_id"].is_unique
assert (final_data["c1_flag"] + final_data["c2_only_flag"] == 1).all()

groups = {
    "C1 — Strict LBP": final_data.loc[final_data["c1_flag"] == 1],
    "C2-only": final_data.loc[final_data["c2_only_flag"] == 1],
}

def summarize_group(df):
    n = len(df)
    transport = df["arrival_transport"].replace("UNKNOWN", np.nan)

    def pct(mask, valid):
        denominator = int(valid.sum())
        return f"{int(mask.sum()):,} ({100 * mask.sum() / denominator:.1f}%)" \
            if denominator else "NA"

    def categorical(col, value):
        valid = df[col].notna()
        return pct(df[col].eq(value), valid)

    rows = {
        "Encounters, n": f"{n:,}",
        "Unique patients, n": f"{df['subject_id'].nunique():,}",
        "Age, mean (SD)": (
            f"{df['estimated_age'].mean():.1f} "
            f"({df['estimated_age'].std():.1f})"
        ),
        "Female, n (%)": categorical("gender", "F"),
        "Ambulance arrival, n (%)": pct(
            transport.eq("AMBULANCE"), transport.notna()
        ),
        "Walk-in arrival, n (%)": pct(
            transport.eq("WALK IN"), transport.notna()
        ),
        "Arrival transport unknown/missing, n":
            f"{int(transport.isna().sum()):,}",
    }

    # Triage acuity distribution
    for level in [1, 2, 3, 4, 5]:
        rows[f"Acuity {level}, n (%)"] = categorical(
            "acuity", level
        )

    rows["Acuity missing, n"] = f"{df['acuity'].isna().sum():,}"

    # Frozen P3 text indicators
    for col in ["cc_explicit_lbp", "cc_broad_back_pain"]:
        rows[f"{col}, n (%)"] = categorical(col, 1)

    return pd.Series(rows)

baseline_table = pd.DataFrame({
    name: summarize_group(df)
    for name, df in groups.items()
})

print("Baseline cohort characteristics")
display(baseline_table)


Baseline cohort characteristics


,C1 — Strict LBP,C2-only
"Encounters, n","9,571","4,722"
"Unique patients, n","7,942","4,274"
"Age, mean (SD)",50.9 (17.7),52.3 (18.3)
"Female, n (%)","5,432 (56.8%)","2,798 (59.3%)"
"Ambulance arrival, n (%)","2,658 (28.3%)","1,409 (31.1%)"
"Walk-in arrival, n (%)","6,694 (71.4%)","3,106 (68.6%)"
"Arrival transport unknown/missing, n",194,196
"Acuity 1, n (%)",205 (2.1%),138 (2.9%)
"Acuity 2, n (%)","1,339 (14.0%)",963 (20.6%)
"Acuity 3, n (%)","6,397 (67.1%)","3,116 (66.5%)"


**Findings**

The baseline comparison identified differences in demographic characteristics, triage acuity, and chief complaint representations between the 9,571 C1 encounters and 4,722 C2-only encounters.

C2-only encounters had a slightly higher mean estimated age (52.3 vs. 50.9 years), female proportion (59.3% vs. 56.8%), and ambulance arrival proportion (31.1% vs. 28.3%).

Triage acuity distributions also differed. Higher-acuity levels (1–2) accounted for 23.5% of C2-only encounters with recorded acuity, compared with 16.1% of C1 encounters. Lower-acuity levels (4–5) were less frequent in C2-only (10.0% vs. 16.8%).

The largest descriptive differences appeared in chief complaint representations. Explicit LBP terminology was identified in 39.1% of C1 encounters but only 6.9% of C2-only encounters. Broad back pain terminology was identified in 77.0% and 57.7%, respectively.

**Interpretation**

These findings suggest that expanding the operational ICD phenotype changes the composition of the analytical population, particularly its triage acuity distribution and presenting complaint characteristics.

The observed differences motivate the planned C1-versus-C2 predictive sensitivity analysis. However, they do not establish the clinical validity of either phenotype or demonstrate differences in opioid dispensing.

The comparison is descriptive, and some patients may contribute encounters to both groups.

### 5.2 Repeat ED Encounters Analysis

**Objective**

Characterize repeated ED encounters among patients in the adult ICD-defined C2 cohort and assess their implications for downstream predictive model evaluation.

**Method**

The number of eligible ED encounters per patient is calculated by grouping the final P4 dataset by `subject_id`.

The analysis summarizes:

- Total encounters and unique patients.
- Mean and maximum encounters per patient.
- Number and proportion of patients with two or more eligible encounters.
- Number and proportion of encounters contributed by repeat-visit patients.
- Distribution of encounters per patient.

Repeated encounters are defined within the study's ICD-defined C2 analytical population, not across all ED visits or the patient's complete clinical history.

**Expected Output**

A concise summary of patient-level encounter frequency and a distribution table describing repeated ED visits.

These results will support patient-level data partitioning in P6 to prevent encounters from the same patient appearing in both training and held-out test sets.

In [11]:

# 5.2 Repeat ED encounters within the adult C2 cohort

visits_per_patient = (
    final_data.groupby("subject_id")["stay_id"]
    .nunique()
)

n_patients = len(visits_per_patient)
n_encounters = len(final_data)

repeat_patients = visits_per_patient >= 2
repeat_patient_n = int(repeat_patients.sum())

repeat_encounter_n = int(
    visits_per_patient[repeat_patients].sum()
)

# Summary statistics
repeat_summary = pd.DataFrame({
    "Measure": [
        "Total ED encounters",
        "Unique patients",
        "Mean encounters per patient",
        "Median encounters per patient",
        "Maximum encounters per patient",
        "Patients with >=2 encounters",
        "Patients with >=2 encounters (%)",
        "Encounters from repeat-visit patients",
        "Encounters from repeat-visit patients (%)"
    ],
    "Value": [
        f"{n_encounters:,}",
        f"{n_patients:,}",
        f"{visits_per_patient.mean():.2f}",
        f"{visits_per_patient.median():.0f}",
        f"{visits_per_patient.max():,}",
        f"{repeat_patient_n:,}",
        f"{repeat_patient_n / n_patients * 100:.1f}%",
        f"{repeat_encounter_n:,}",
        f"{repeat_encounter_n / n_encounters * 100:.1f}%"
    ]
})

# Encounter frequency distribution
frequency = visits_per_patient.value_counts().sort_index()

visit_distribution = pd.DataFrame({
    "encounters_per_patient": frequency.index,
    "patients": frequency.values
})

visit_distribution["patient_pct"] = (
    visit_distribution["patients"] / n_patients * 100
).round(2)

# Integrity checks
assert visits_per_patient.sum() == n_encounters
assert visit_distribution["patients"].sum() == n_patients

print("Repeat encounter analysis: PASSED")
display(repeat_summary)
display(visit_distribution)


Repeat encounter analysis: PASSED


,Measure,Value
0,Total ED encounters,"14,293"
1,Unique patients,"11,404"
2,Mean encounters per patient,1.25
3,Median encounters per patient,1
4,Maximum encounters per patient,34
5,Patients with >=2 encounters,"1,705"
6,Patients with >=2 encounters (%),15.0%
7,Encounters from repeat-visit patients,"4,594"
8,Encounters from repeat-visit patients (%),32.1%


,encounters_per_patient,patients,patient_pct
0,1,9699,85.05
1,2,1166,10.22
2,3,309,2.71
3,4,114,1.00
4,5,40,0.35
5,6,30,0.26
6,7,16,0.14
7,8,5,0.04
8,9,3,0.03
9,10,4,0.04


**Findings**

The adult C2 cohort included 14,293 eligible ED encounters among 11,404 unique patients, with a mean of 1.25 encounters per patient and a median of one encounter.

Most patients (9,699; 85.0%) contributed only one eligible encounter. A total of 1,705 patients (15.0%) had two or more encounters, accounting for 4,594 encounters (32.1% of the analytical population).

The maximum observed number of eligible encounters for a single patient was 34.

**Interpretation**

Repeated ED encounters introduce patient-level dependence within the analytical dataset. Although repeat-visit patients represent a minority of individuals, they contribute nearly one-third of all eligible encounters.

These findings support the prespecified patient-level data partitioning strategy in P6. All encounters belonging to the same `subject_id` will be assigned to the same training or test partition, with consistent patient assignments across C1 and C2 experiments.

The observed encounter frequencies describe repeated visits within the ICD-defined C2 cohort and do not represent patients' complete ED utilization histories.

No opioid outcomes or clinical conclusions about repeat utilization are inferred from this analysis.

## 6. Final Dataset Contract & P5 Handoff

### 6.1 Final Analytical Dataset

**Output:** `results/04_cohort_integration/adult_lbp_encounter_features.csv`

P4 produced a validated encounter-level analytical dataset containing **14,293 adult ED encounters, 11,404 unique patients, and 23 columns**.

The dataset contains two nested ICD-defined LBP cohorts:

| Cohort | Encounters | Unique Patients |
|---|---:|---:|
| C1 — Strict LBP | 9,571 | 7,942 |
| C2-only | 4,722 | 4,274 |
| C2 — Expanded LBP | 14,293 | 11,404 |

Six encounters were excluded from the initial ICD-defined population because adult eligibility could not be established using linked patient information. All retained encounters had estimated ages ≥18 years.

C1 is the primary analytical cohort. C2 includes C1 and the additional C2-only encounters, supporting the planned phenotype sensitivity analysis.

### 6.2 Data Integrity and Feature Contract

The final integrated dataset passed all implemented integrity checks:

- One record per `stay_id`, without missing encounter or patient identifiers.
- Valid adult eligibility and consistent nested C1/C2 membership.
- Complete structured triage and P3 chief complaint feature linkage.
- No encounter loss or row multiplication during integration.
- Successful CSV export and reload verification, including explicit data types for key fields.

**Final feature contract**

| Feature Group | Variables | Downstream Use |
|---|---|---|
| Identifiers | `stay_id`, `subject_id` | Encounter linkage; patient-level splitting |
| Cohort indicators | `c1_flag`, `c2_flag`, `c2_only_flag` | Cohort selection and sensitivity analysis only |
| Demographics | `estimated_age`, `gender` | Candidate predictors |
| Core triage | `acuity`, `pain` | Candidate predictors |
| Extended triage | `arrival_transport`, `heartrate`, `sbp`, `dbp`, `resprate`, `temperature`, `o2sat` | Candidate predictors |
| Chief complaint text | `chiefcomplaint`, `cc_text_normalized` | Text analysis and representation |
| P3 clinical indicators | `cc_text_missing`, `cc_explicit_lbp`, `cc_sciatica`, `cc_broad_back_pain`, `cc_radiating_leg_pain` | Rule-based text predictors |

The 23-column dataset retains raw clinical information without outcome-based feature selection.

ICD-derived cohort indicators and patient/encounter identifiers will not be included in the predictive feature matrix.

Raw pain values, including nonnumeric expressions, are preserved as text. Learned imputation, categorical encoding, and statistical text transformations will be fitted exclusively on training data in P6.

### 6.3 Cohort Characteristics and Methodological Considerations

**Cohort composition**

Section 5.1 identified descriptive differences between C1 and C2-only encounters.

C2-only encounters had a higher proportion of triage acuity levels 1–2 (23.5% vs. 16.1%) and a substantially lower frequency of explicit LBP terminology in chief complaints (6.9% vs. 39.1%).

These findings indicate differences in the composition of the two operational ICD-defined populations and provide context for the planned phenotype sensitivity analysis.

They do not establish the clinical validity of either phenotype or explain opioid dispensing behavior.

**Repeated encounters**

Section 5.2 identified 1,705 patients (15.0%) with at least two eligible ED encounters. These patients contributed 4,594 encounters (32.1% of the analytical dataset).

Patient-level data partitioning will therefore be used in P6, with all encounters from the same `subject_id` assigned to the same partition across C1 and C2 experiments.

**Methodological limitations**

- Final ICD diagnoses identify the analytical population retrospectively and are not available at ED triage.
- Estimated age is derived from MIMIC-IV deidentification anchors rather than exact birth dates.
- The available triage data do not establish precise measurement timestamps for every candidate predictor.
- Chief complaint indicators are frozen, rule-based text representations rather than independently validated clinical phenotypes.
- The final dataset contains no opioid outcome labels.

The analysis is therefore a retrospective conditional prediction study rather than a prospectively deployable ED triage prediction system.

### 6.4 P5 Handoff

P5 will construct the binary outcome of **documented opioid dispensing during the ED encounter** using MIMIC-IV-ED Pyxis records.

The outcome construction will:

1. Develop and review a reproducible opioid medication mapping.
2. Define the qualifying ED dispensing time window.
3. Identify qualifying dispensing events and aggregate them to one binary indicator per `stay_id`.
4. Left join the outcome indicators to the frozen P4 analytical dataset.
5. Validate encounter preservation, medication mapping coverage, and outcome prevalence.

Encounters without a qualifying Pyxis record will be retained and labeled as having no *observed* qualifying opioid dispensing event, rather than being excluded.

The outcome will represent documented dispensing, not confirmed administration, discharge prescribing, opioid consumption, or treatment appropriateness.

P6 will subsequently apply patient-level data partitioning, training-only feature preprocessing, and predictive model evaluation.

### 6.5 Completion Statement

**P4 — Encounter-Level Cohort Integration: Completed.**

The validated analytical dataset provides a reproducible foundation for opioid outcome construction and predictive modeling within two nested ICD-defined LBP cohorts.

The P4 cohort definitions, eligibility criteria, and frozen P3 text rules will remain unchanged during downstream modeling.

Remaining project phases will address opioid outcome construction (P5), predictive modeling (P6), explainability and phenotype sensitivity (P7), and final reporting (P8).